In [1]:
import numpy as np
import sklearn
import torch
import os
import pandas as pd
import tqdm
import random

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
import torchmetrics

from transformers import AutoTokenizer

/home/cam/miniforge3/envs/jupyter_dl/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = torch.device('cpu' if not torch.cuda.is_available() else 'cuda')
device

device(type='cuda')

In [3]:
if not os.path.exists('IMDB-Dataset.csv'):
  !wget -O IMDB-Dataset.csv -q "https://www.dropbox.com/scl/fi/0c7zc2adk1mgwgut5w80w/IMDB-Dataset.csv?rlkey=1drfg4zw36mhu32ndy2ihnygw&dl=1"

In [4]:
df = pd.read_csv('IMDB-Dataset.csv')
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [5]:
text = list(df['review'].str.replace('<br />',''))
labels = np.array(df['sentiment'].map({'negative':0,'positive':1}))

In [6]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

Example of how to tokenize text:


In [7]:
seq = text[0][:10]
seq

'One of the'

In [8]:
token_ids = tokenizer(seq)['input_ids']
token_ids

[101, 1448, 1104, 1103, 102]

In [9]:
tokenizer.decode(token_ids+[0,0,0])

'[CLS] One of the [SEP] [PAD] [PAD] [PAD]'

## 1. Bag of words model

In [16]:
test_size = 0.10
tf_idf = TfidfVectorizer()
X_train, X_test, y_train, y_test = train_test_split(tf_idf.fit_transform(text), labels, test_size=test_size)

In [17]:
X_train.shape[0]

45000

In [13]:
model = MLPClassifier()
model.fit(X_train, y_train)

MLPClassifier()

In [14]:
print(f'Train score:{model.score(X_train, y_train)}')
print(f'Test score:{model.score(X_test, y_test)}')

Train score:1.0
Test score:0.8658


## 2. RNN Model

In [18]:
# Initialize BPE tokenizer pretrained for GPT2
tokenizer = AutoTokenizer.from_pretrained("gpt2", truncation=True)
tokenizer.pad_token = " "
vocab_size = tokenizer.vocab_size
token_size = tokenizer.model_max_length

In [19]:
# Encode text
tokens = []
for t in text:
    tokenized = tokenizer.encode(t, add_special_tokens=True)
    
    # Apply random truncation only if needed
    if len(tokenized) > token_size:
        idx = random.randint(0, len(t) - token_size)  # Truncate randomly but not too aggressively
        tokenized = tokenizer.encode(t[idx: idx+token_size])

    # Pad/truncate to final size
    tokenized = tokenized[:token_size] + [tokenizer.pad_token_id] * max(0, token_size - len(tokenized))
    
    tokens.append(torch.tensor(tokenized, dtype=torch.float32, device=device))

tokens = torch.stack(tokens)  # Convert list to tensor
tokens.shape

Token indices sequence length is longer than the specified maximum sequence length for this model (1051 > 1024). Running this sequence through the model will result in indexing errors


torch.Size([50000, 1024])

In [24]:
X_train, X_test, y_train, y_test = train_test_split(tokens, torch.tensor(labels, device=device, dtype=torch.float), test_size=test_size)

dataset = torch.utils.data.TensorDataset(X_train, y_train)
train_loader = torch.utils.data.DataLoader(dataset, batch_size=32)

dataset = torch.utils.data.TensorDataset(X_test, y_test)
test_loader = torch.utils.data.DataLoader(dataset, batch_size=512)

In [43]:
import torch
import torch.nn as nn

class NewGRU(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size, num_layers):
        super(NewGRU, self).__init__()
        self.device = device
        self.embedding = nn.Embedding(vocab_size, embed_size, device=device)
        self.gru = nn.GRU(
            input_size=embed_size, 
            hidden_size=hidden_size, 
            num_layers=num_layers,
            batch_first=True,
            device=device
        )
        self.fc = nn.Linear(hidden_size, 1, device=device)  # Output 1 value for binary classification
        self.out = nn.Sigmoid()  # map 1 value to range [0, 1]
        self.to(device)
        
    def forward(self, x):
        x = x.long()  # Ensure input is of type torch.long
        x = self.embedding(x)  # Shape: (batch_size, seq_len, embed_size)
        gru_out, _ = self.gru(x)  # Shape: (batch_size, seq_len, hidden_size)
    
        last_hidden = gru_out[:, -1, :]  # Take last hidden state
        logits = self.fc(last_hidden)  # Shape: (batch_size, 1)
        logits = logits.reshape(logits.shape[0])  # Shape: (batch_size, 1)
        return self.out(logits)

model = NewGRU(vocab_size, 100, 100, 3)
opt = torch.optim.SGD(model.parameters(), lr=3e-4)
criterion = nn.BCEWithLogitsLoss()

In [44]:
epochs = 10
threshold = 0.5
print('Training loss')

for epoch in range(epochs):
    correct = 0
    for X, y in train_loader:
        opt.zero_grad()
        z = model(X)
        loss = criterion(z, y)
        loss.backward()
        opt.step()
        
        correct += sum((z > threshold) == y)
        
    print(f'[{epoch}] {loss}')

train_acc = correct / X_train.shape[0]

Training loss
[0] 0.84514319896698
[1] 0.8340209722518921
[2] 0.8237687945365906
[3] 0.814260721206665
[4] 0.805419921875
[5] 0.7972265481948853
[6] 0.7896586656570435
[7] 0.7827073335647583
[8] 0.7763723731040955
[9] 0.7706176042556763


In [58]:
# Calculate test accuracy
model.eval()
correct = 0
with torch.no_grad():
    for X, y in test_loader:
        z = model(X)
        correct += sum((z > threshold) == y)
        
test_acc = correct / X_test.shape[0]

In [59]:
print(f'Train score: {train_acc}')
print(f'Test score: {test_acc}')

Train score: 0.49915555119514465
Test score: 0.5076000094413757


In [60]:
model(X_train[:20])

tensor([0.2727, 0.2727, 0.2727, 0.2727, 0.2727, 0.2727, 0.2727, 0.2727, 0.2727,
        0.2727, 0.2727, 0.2727, 0.2727, 0.2727, 0.2727, 0.2727, 0.2727, 0.2727,
        0.2727, 0.2727], device='cuda:0', grad_fn=<SigmoidBackward0>)